# 02 — Ablations rewrite / rerank + latence e2e

Étude d'ablation typique d'un pipeline RAG avancé :
1. Qualité des chunks (Hit Rate / MRR) avec/sans query rewrite et re-ranking
2. Latence end-to-end du pipeline `ask()` (retrieval + génération)

**Coût** : appelle l'API OpenAI (rewrite, rerank, génération).

In [ ]:
from pathlib import Path
import sys

import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from eval.performance_study import (
    evaluate_advanced_config,
    evaluate_e2e_latency,
    run_performance_study,
)
from eval.retrieval_eval import load_ground_truth
from rag.config import get_settings

settings = get_settings()
queries = load_ground_truth()
TOP_K = 5
E2E_LIMIT = 5
print(f"{len(queries)} questions | e2e_limit={E2E_LIMIT}")

## Ablations (qualité des sources)

In [ ]:
ablation_rows = []
for use_rewrite, use_rerank in [
    (False, False), (True, False), (False, True), (True, True)
]:
    print(f"rewrite={use_rewrite} rerank={use_rerank}")
    row = evaluate_advanced_config(
        queries, TOP_K, settings,
        use_rewrite=use_rewrite,
        use_rerank=use_rerank,
    )
    ablation_rows.append({
        "config": row["config"],
        "hit_rate": row["metrics"]["hit_rate"],
        "mrr": row["metrics"]["mrr"],
        "mean_ms": row["latency"]["mean_ms"],
        "p95_ms": row["latency"]["p95_ms"],
    })

df_abl = pd.DataFrame(ablation_rows)
df_abl

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
df_abl.plot(x="config", y=["hit_rate", "mrr"], kind="bar", ax=axes[0], rot=25)
axes[0].set_ylim(0, 1.05)
axes[0].set_title("Ablations — qualité")

df_abl.plot(
    x="config", y="mean_ms", kind="bar", ax=axes[1], rot=25,
    color="#54A24B", legend=False,
)
axes[1].set_title("Ablations — latence moyenne")
axes[1].set_ylabel("ms")
plt.tight_layout()
plt.show()

## Latence end-to-end

In [ ]:
sample = [q.question for q in queries[:E2E_LIMIT]]
e2e_rows = []
for rewrite, rerank in [(False, False), (True, True)]:
    print(f"e2e rewrite={rewrite} rerank={rerank}")
    row = evaluate_e2e_latency(
        sample, settings, rewrite=rewrite, rerank=rerank, top_k=TOP_K
    )
    e2e_rows.append({
        "config": row["config"],
        "n": row["num_questions"],
        "mean_ms": row["latency"]["mean_ms"],
        "p50_ms": row["latency"]["p50_ms"],
        "p95_ms": row["latency"]["p95_ms"],
    })

df_e2e = pd.DataFrame(e2e_rows)
df_e2e

## Étude complète (JSON + Markdown)

In [ ]:
# Relance l'étude officielle (écrit eval/results/ + eval/ETUDE_PERFORMANCE.md)
# Décommenter pour tout régénérer d'un coup :
# report = run_performance_study(top_k=TOP_K, e2e_limit=E2E_LIMIT)
# print(report["best_retrieval_mode"], report["best_advanced_config"])
# for c in report["conclusions"]:
#     print("-", c)

print("Astuce : python scripts/run_performance_study.py -v")